# Blood-brain barrier permeability prediction by RDKit and Random Forest
A minimal, reproducible pipeline that predicts whether a small molecule can penetrate the blood-brain barrier (BBB) using Morgan (ECFP4) fingerprints and a Random Forest Classifier.

Dataset: BBBP (Blood–Brain Barrier Penetration) from [DeepChem](https://deepchem.io/).

## 1. Setup & Load data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, confusion_matrix, classification_report

os.makedirs("figures", exist_ok=True) #Output directory for saved figures

url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(url)
print(f"{df.shape[0]} molecules; {df.shape[1]} fields/columns")
print("columns:", list(df.columns))
df.head()

This dataset contains the information of 2050 small molecules, including the name, BBB penetration ability (p_np, 0=unable, 1=able), and SMILES string.

## 2. Exploratory data analysis

In [ ]:
print("target distribution")
print(df["p_np"].value_counts())
print("positive samples ratio:", round(df["p_np"].mean()*100, 2), "%")

**Class imbalance:** The target distribution demonstrates a very high positive samples ratio, 76.44%, indicating the imbalance of the two classes, which is the root of the problems later.

## 3. Future engineering -- Morgan fingerprints
Machine learning models are unable to read and interpret molecular graphs directly, so each SMILES string should be encoded into a binary vector which can be learnt by computers.

ECPR4 (Morgan circular fingerprint, radius=2, nbits=2048) is applied to encode molecular sub-structure. SMILES strings are computed to 2048-dimensional binary vectors (1=sub-structure present), assembling the feature matrix X for Random Forest Classifier.

In [ ]:
def smiles_to_fingerprint(smiles, radius=2, nbits=2048): #radius=2:ECFP4指纹，每个原子向外搜索2根化学键的局部结构； nbit通常取1024/2048
    """convert a SMILE str into Morgan fingerprint (bit vector)"""
    mol = Chem.MolFromSmiles(smiles) #SMILE str --> molecular object
    if mol is None: #SMILES格式非法时返回None，作判空处理
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits) #生成二进制Morgan指纹（bit vector），每一位0/1代表种子结构是否存在
    arr = np.zeros((1,))
    DataStructs.ConvertToNumpyArray(fp, arr) #指纹对象不能直接用于sklearn，先把bit-vector拷贝到numpy数组
    return arr #返回numpy一维数组，形状（2048，），元素只有0、1

test_smiles = df.iloc[0]["smiles"] #从pandas表格取第一条分子SMILES
test_fp = smiles_to_fingerprint(test_smiles)
print(f"length of fingerprint: {len(test_fp)}")
print(f"places of 1 in fingerprint: {int(test_fp.sum())}") #代表该分子识别到多少种子结构特征

fingerprints = [smiles_to_fingerprint(s) for s in df["smiles"]] #遍历df全部smiles列，对每一条用指纹转化函数
valid_idx = [i for i, fp in enumerate(fingerprints) if fp is not None] #enumerate拿到每条指纹的原始行号i，收集转化成功的样本存入valid_idx

X = np.array([fingerprints[i] for i in valid_idx]) #根据有效下标取出指纹数组，组装成numpy二维特征矩阵
y = df["p_np"].values[valid_idx] #p_np是数据集的特征属性（这里指BBB穿透能力），取出转化为numpy一维数组，同样用valid_idx作索引
print(f"the shape of the feature matrix X: {X.shape}" ) #2039行（有效样本数）× 2048列（每个分子的2018位Morgan指纹）；每一列：一个子结构的特征bit
print(f"the shape of the target y: {y.shape}") #2039行 × 1列

## 4. Model training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
#stratify=y: 分层抽样，让训练集和测试集里面白浅类别占比和原始数据保持一致
model = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=1)
#n_jobs=1: 只用单个CPU核心，训练慢，适合小数据集调试，防止多进程冲突
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test) [:,1]#预测X_test可能的分组

**"stratify=y"** implements stratified sampling during dataset spliting. It adjusts the class proportion in both training and test set accoring to the original label "y", preveting imbalanced class distribution and reducing bias during model evaluation.

## 5. Evaluation

In [ ]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_prob):.3f}")
print(classification_report(y_test, y_pred, target_names=["unable to penetrate BBB","able to penetrate BBB"]))

**Key insight:** Although the results "Accuracy=0.868, ROC AUC=0.911" seem to be highly satisfactory, the recall of molecules unable to penetrate BBB (recall=0.51) suggests some bias of the model towards "positive sample". Among the actual negaive samples, about half of them are mistakenly identified as positive ones, which may increase the burden on wet lab verification. Additionally, since the positive samples ratio is 76.44%, the model can reach this accuracy just by deciding the "y_test" all positive. Actually, the accuracy (86.8%) is only a little higher than 76.44%, indicating the illusory strength of the model.    

## 6. Visualization

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, label=f"Random Forest (AUC={roc_auc_score(y_test, y_prob):.3f})")
plt.plot([0,1],[0,1], 'k--', label="random guess")
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.legend()
plt.title("ROC Curve")
plt.tight_layout()
plt.savefig(".ipynb_checkpoints", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
cm = confusion_matrix(y_test, y_pred) #行：真实标签；列：预测标签；对角线：预测正确的样本
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["non-pen", "pen"], yticklabels=["non-pen", "pen"]) #annot=True: 每个格子显示数字；fmt=‘d'：格式为整数
plt.title("Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(".ipynb_checkpoints", dpi=150, bbox_inches="tight")
plt.show()

## 7. Prediction on new molecules 

In [ ]:
caffeine_smiles = "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"
caffeine_fp = smiles_to_fingerprint(caffeine_smiles)
prob = model.predict_proba(caffeine_fp.reshape(1,-1))[0,1] #reshape(1,-1): (2048,)变成（1,2048），-1代表numpy自动算特征维度
#predict_proba返回shape(n_samples, n_classes), 这里输出(1,2), 第0列：不能（0）的概率，第1列：能（1）的概率。
#[0,1]: 0:取第1个（唯一）样本；1：取类别1（能穿透）的概率值，0~1的浮点数
print(f"the percentage of caffeine penetrating the BBB: {prob:.2%}") #转化为百分比，保留2位小数